In [83]:
import polars as pl
kpler = pl.read_parquet(r"c:\Users\HanhTa\Downloads\power_loads_forecasts 1 (1).parquet")
kpler = kpler.sort("time")
print(kpler)

shape: (3_511_200, 10)
┌──────────────┬──────┬─────────┬─────┬───┬────────────┬──────────────┬──────────────┬─────────────┐
│ time         ┆ zone ┆ model   ┆ run ┆ … ┆ runDate    ┆ updatedAt    ┆ load_type    ┆ retrieved_a │
│ ---          ┆ ---  ┆ ---     ┆ --- ┆   ┆ ---        ┆ ---          ┆ ---          ┆ t           │
│ datetime[μs, ┆ str  ┆ str     ┆ str ┆   ┆ str        ┆ str          ┆ str          ┆ ---         │
│ UTC]         ┆      ┆         ┆     ┆   ┆            ┆              ┆              ┆ datetime[μs │
│              ┆      ┆         ┆     ┆   ┆            ┆              ┆              ┆ , UTC]      │
╞══════════════╪══════╪═════════╪═════╪═══╪════════════╪══════════════╪══════════════╪═════════════╡
│ 2026-01-01   ┆ DE   ┆ GFS_ENS ┆ 00z ┆ … ┆ 2026-01-01 ┆ null         ┆ residual_dem ┆ 2026-09-24  │
│ 00:00:00 UTC ┆      ┆         ┆     ┆   ┆            ┆              ┆ and          ┆ 07:57:30.10 │
│              ┆      ┆         ┆     ┆   ┆            ┆            

In [84]:
print(kpler.columns)

['time', 'zone', 'model', 'run', 'value', 'provider', 'runDate', 'updatedAt', 'load_type', 'retrieved_at']


In [85]:
kpler_data = kpler.filter(
    pl.col("time").dt.date()
    ==
    pl.col("runDate").str.to_datetime(time_zone="UTC").dt.date()
)
print(kpler_data)

shape: (177_888, 10)
┌─────────────┬──────┬─────────────┬─────┬───┬────────────┬─────────────┬─────────────┬────────────┐
│ time        ┆ zone ┆ model       ┆ run ┆ … ┆ runDate    ┆ updatedAt   ┆ load_type   ┆ retrieved_ │
│ ---         ┆ ---  ┆ ---         ┆ --- ┆   ┆ ---        ┆ ---         ┆ ---         ┆ at         │
│ datetime[μs ┆ str  ┆ str         ┆ str ┆   ┆ str        ┆ str         ┆ str         ┆ ---        │
│ , UTC]      ┆      ┆             ┆     ┆   ┆            ┆             ┆             ┆ datetime[μ │
│             ┆      ┆             ┆     ┆   ┆            ┆             ┆             ┆ s, UTC]    │
╞═════════════╪══════╪═════════════╪═════╪═══╪════════════╪═════════════╪═════════════╪════════════╡
│ 2026-01-01  ┆ DE   ┆ GFS_ENS     ┆ 00z ┆ … ┆ 2026-01-01 ┆ null        ┆ residual_de ┆ 2026-09-24 │
│ 00:00:00    ┆      ┆             ┆     ┆   ┆            ┆             ┆ mand        ┆ 07:57:30.1 │
│ UTC         ┆      ┆             ┆     ┆   ┆            ┆           

In [86]:
# kpler with models  value by pivoting

kpler_data = kpler_data.pivot(on="model",index="time",values="value")
print(kpler_data)

shape: (25_632, 8)
┌───────────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬─────────────┐
│ time          ┆ GFS_ENS  ┆ EC_OP    ┆ EC_46    ┆ GFS_OP   ┆ EC_AIFS  ┆ EC_ENS   ┆ EC_AIFS_ENS │
│ ---           ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---         │
│ datetime[μs,  ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64      ┆ f64         │
│ UTC]          ┆          ┆          ┆          ┆          ┆          ┆          ┆             │
╞═══════════════╪══════════╪══════════╪══════════╪══════════╪══════════╪══════════╪═════════════╡
│ 2026-01-01    ┆ 6870.44  ┆ 6489.05  ┆ 10183.42 ┆ 3739.45  ┆ 8379.26  ┆ 7036.98  ┆ null        │
│ 00:00:00 UTC  ┆          ┆          ┆          ┆          ┆          ┆          ┆             │
│ 2026-01-01    ┆ 6180.66  ┆ 6216.17  ┆ 10183.42 ┆ 3088.67  ┆ 7693.09  ┆ 6318.96  ┆ null        │
│ 00:15:00 UTC  ┆          ┆          ┆          ┆          ┆          ┆          ┆             │
│

In [87]:
kpler_data.describe()

statistic,time,GFS_ENS,EC_OP,EC_46,GFS_OP,EC_AIFS,EC_ENS,EC_AIFS_ENS
str,str,f64,f64,f64,f64,f64,f64,f64
"""count""","""25632""",25632.0,25632.0,25536.0,25632.0,25632.0,25632.0,24192.0
"""null_count""","""0""",0.0,0.0,96.0,0.0,0.0,0.0,1440.0
"""mean""","""2026-05-14 11:52:30+00:00""",23275.242675,25176.715214,26471.091213,23486.240233,27130.777049,24954.661049,26812.051737
"""std""",null,19538.571723,18130.119707,17552.988109,19694.584337,17647.644338,17864.414301,17336.855598
"""min""","""2026-01-01 00:00:00+00:00""",-53624.98,-51661.66,-46340.0,-60886.52,-45303.91,-48711.21,-44168.17
"""25%""","""2026-03-08 18:00:00+00:00""",10574.5,13800.86,15358.51,10908.68,16026.1,13702.27,15724.75
"""50%""","""2026-05-14 12:00:00+00:00""",27348.65,28655.91,29882.87,27754.99,30544.58,28266.68,30219.61
"""75%""","""2026-07-20 05:45:00+00:00""",37607.77,38239.71,39082.94,37852.07,39848.96,37814.85,39386.38
"""max""","""2026-09-24 23:45:00+00:00""",68202.8,65265.58,65792.51,67876.24,69074.56,64922.23,68440.23


In [88]:
kpler_data.write_parquet("kpler_data.parquet")